# Pandas — Airbnb Dublin Analysis

Benchmark of data processing operations on the **Dublin Airbnb dataset** using Pandas.

| File | Description |
|------|-------------|
| `listings.csv` | ~7,000 listings — price, location, ratings |
| `calendar.csv.gz` | ~2.5M rows — daily availability and pricing |

**Pipeline:** Load → Clean → Aggregate calendar → Join → Group by neighbourhood × room type

To run with the **large dataset** (5× replicated), start the container with:
```
docker compose run --rm -e LISTINGS_FILE=listings_large.csv -e CALENDAR_FILE=calendar_large.csv.gz pandas jupyter lab ...
```

In [ ]:
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_PATH     = "/data"
timings = {}
total_start = time.perf_counter()
print("Ready.")

In [ ]:
# Dataset selection — override with env vars to switch to the large dataset
LISTINGS_FILE = os.environ.get("LISTINGS_FILE", "listings.csv")
CALENDAR_FILE = os.environ.get("CALENDAR_FILE", "calendar.csv.gz")
print(f"Listings : {DATA_PATH}/{LISTINGS_FILE}")
print(f"Calendar : {DATA_PATH}/{CALENDAR_FILE}")

## 1. Load Data

Pandas reads both files in a single pass. `calendar.csv.gz` is decompressed on-the-fly.

In [ ]:
t = time.perf_counter()
listings_raw = pd.read_csv(f"{DATA_PATH}/{LISTINGS_FILE}", low_memory=False)
calendar_raw = pd.read_csv(f"{DATA_PATH}/{CALENDAR_FILE}", low_memory=False)
timings["load_data"] = time.perf_counter() - t

print(f"Listings : {len(listings_raw):,} rows × {listings_raw.shape[1]} cols  ({timings['load_data']:.3f}s)")
print(f"Calendar : {len(calendar_raw):,} rows × {calendar_raw.shape[1]} cols")
display(listings_raw[["id","neighbourhood_cleansed","room_type","price","number_of_reviews","review_scores_rating"]].head(3))

## 2. Clean Listings

Select relevant columns, parse the `price` field (strip `$` and `,`), drop rows with no valid price.

In [ ]:
t = time.perf_counter()
listings = listings_raw[[
    "id", "neighbourhood_cleansed", "room_type",
    "price", "number_of_reviews", "review_scores_rating", "accommodates",
]].copy()
listings.rename(columns={"id": "listing_id"}, inplace=True)
listings["listing_id"] = pd.to_numeric(listings["listing_id"], errors="coerce")
listings["price"] = (
    listings["price"].astype(str)
    .str.replace(r"[\$,]", "", regex=True)
    .pipe(pd.to_numeric, errors="coerce")
)
listings = listings[listings["price"].notna() & (listings["price"] > 0)]
timings["clean_listings"] = time.perf_counter() - t

print(f"Cleaned listings: {len(listings):,} rows  ({timings['clean_listings']:.3f}s)")
display(listings.head(3))

## 3. Clean Calendar

Convert `available` (t/f) to binary, parse the `price` column.

In [ ]:
t = time.perf_counter()
calendar = calendar_raw[["listing_id", "date", "available", "price"]].copy()
calendar["listing_id"] = pd.to_numeric(calendar["listing_id"], errors="coerce")
calendar["is_available"] = (calendar["available"] == "t").astype(int)
calendar["price"] = (
    calendar["price"].astype(str)
    .str.replace(r"[\$,]", "", regex=True)
    .pipe(pd.to_numeric, errors="coerce")
)
timings["clean_calendar"] = time.perf_counter() - t

print(f"Cleaned calendar: {len(calendar):,} rows  ({timings['clean_calendar']:.3f}s)")
display(calendar.head(3))

## 4. Aggregate Calendar Stats per Listing

For each listing compute:
- `total_days` — days in calendar window
- `days_available` — days marked available (not booked)
- `avg_booked_price` — mean price on booked days
- `occupancy_rate` — (total − available) / total

In [ ]:
t = time.perf_counter()
booked_price = (
    calendar[calendar["is_available"] == 0]
    .groupby("listing_id")["price"]
    .mean()
    .rename("avg_booked_price")
)
cal_agg = calendar.groupby("listing_id").agg(
    total_days=("is_available", "count"),
    days_available=("is_available", "sum"),
).join(booked_price)
cal_agg["occupancy_rate"] = (
    (cal_agg["total_days"] - cal_agg["days_available"]) / cal_agg["total_days"]
)
cal_agg = cal_agg.reset_index()
timings["calendar_agg"] = time.perf_counter() - t

print(f"Calendar aggregates: {len(cal_agg):,} listings  ({timings['calendar_agg']:.3f}s)")
display(cal_agg.head(3))

## 5. Join Listings with Calendar Aggregates

Left-join so listings without calendar data are retained.

In [ ]:
t = time.perf_counter()
joined = listings.merge(cal_agg, on="listing_id", how="left")
timings["join"] = time.perf_counter() - t

print(f"Joined: {len(joined):,} rows × {joined.shape[1]} cols  ({timings['join']:.3f}s)")
display(joined[["listing_id","neighbourhood_cleansed","room_type","price","occupancy_rate","avg_booked_price"]].head(3))

## 6. Neighbourhood × Room Type Analysis

Group by `neighbourhood_cleansed` + `room_type`, compute aggregate stats, sort by listing count.

In [ ]:
t = time.perf_counter()
neighbourhood_stats = (
    joined
    .groupby(["neighbourhood_cleansed", "room_type"], dropna=False)
    .agg(
        listing_count=("listing_id", "count"),
        avg_price=("price", "mean"),
        avg_occupancy=("occupancy_rate", "mean"),
        avg_rating=("review_scores_rating", "mean"),
    )
    .sort_values("listing_count", ascending=False)
    .reset_index()
)
timings["neighbourhood_agg"] = time.perf_counter() - t

top20 = neighbourhood_stats.head(20).copy()
top20["avg_price"]     = top20["avg_price"].round(2)
top20["avg_occupancy"] = (top20["avg_occupancy"] * 100).round(1).astype(str) + "%"
top20["avg_rating"]    = top20["avg_rating"].round(2)
top20.columns = ["Neighbourhood", "Room Type", "Count", "Avg Price (€)", "Occupancy", "Rating"]
print(f"Computed in {timings['neighbourhood_agg']:.3f}s")
display(top20.style.background_gradient(subset=["Count"], cmap="Blues").format(precision=2))

## 7. Room Type Summary

In [ ]:
t = time.perf_counter()
room_type_stats = (
    joined
    .groupby("room_type", dropna=False)
    .agg(
        count=("listing_id", "count"),
        avg_price=("price", "mean"),
        avg_occupancy=("occupancy_rate", "mean"),
        avg_rating=("review_scores_rating", "mean"),
    )
    .sort_values("count", ascending=False)
    .reset_index()
)
timings["room_type_agg"] = time.perf_counter() - t
timings["total"] = time.perf_counter() - total_start

display_rt = room_type_stats.copy()
display_rt["avg_price"]     = display_rt["avg_price"].round(2)
display_rt["avg_occupancy"] = (display_rt["avg_occupancy"] * 100).round(1).astype(str) + "%"
display_rt["avg_rating"]    = display_rt["avg_rating"].round(2)
display_rt.columns = ["Room Type", "Count", "Avg Price (€)", "Occupancy", "Rating"]
display(display_rt.style.bar(subset=["Count"], color="#4472C4", width=80))

## 8. Timing Summary

In [ ]:
steps = {k: v for k, v in timings.items() if k != "total"}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(list(steps.keys()), list(steps.values()), color="#4472C4", edgecolor="white")
ax.bar_label(bars, fmt="%.3fs", padding=5, fontsize=10)
ax.set_xlabel("Seconds")
ax.set_title(f"Pandas — Step Timing  (total: {timings['total']:.3f}s)", fontsize=13, fontweight="bold")
ax.invert_yaxis()
ax.set_xlim(0, max(steps.values()) * 1.25)
plt.tight_layout()
plt.show()

print(f"\nDataset : {LISTINGS_FILE}  |  {CALENDAR_FILE}")
print(f"Total wall-clock time: {timings['total']:.3f}s")